# RAG from a Live News Web Page — **Qwen3-4B (4-bit)** on **Kaggle**

Same classroom demo as the Colab notebook (*LLM alone* vs *LLM + retrieved web text*),
but here we test whether **`Qwen/Qwen3-4B` loaded in 4-bit** gives good enough answers
and speed on a **free Kaggle GPU**.

* **Event:** glacier collapse and flooding in Nepal, August 2026.
* **Everything is LangChain:** `WebBaseLoader → RecursiveCharacterTextSplitter →
  HuggingFaceEmbeddings → FAISS → retriever → ChatPromptTemplate → LCEL chain`.
* **Model:** `Qwen/Qwen3-4B`, 4-bit NF4 quantization via `bitsandbytes`.

---

## ⚙️ Kaggle setup — do this first (right-hand **Settings** panel)

1. **Accelerator → GPU** (pick **GPU T4 x2** or **GPU P100** — both are free).
2. **Internet → On.**  *Required* — we pip-install packages, download the model from
   Hugging Face, and fetch a news page. With Internet off, nothing below works.
3. (No Hugging Face token needed — `Qwen/Qwen3-4B` is a public model.)

> The model weights are ~8 GB. They download once per session into the (temporary)
> cache, then `bitsandbytes` quantizes them to ~3 GB in GPU memory as they load.

## 1. Install / upgrade the libraries

Kaggle already has `torch`, `transformers`, `accelerate`, `bitsandbytes`, but the
`transformers` version is often **too old for the Qwen3 architecture** (needs
`>= 4.51`), so we upgrade those four. The LangChain packages are not preinstalled.

In [ ]:
# --- LLM stack: upgrade so the Qwen3 architecture is recognised ---
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes

# --- LangChain stack (core primitives only; no big `langchain` meta-package) ---
!pip install -q langchain-core langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers beautifulsoup4

# print what actually got installed - makes any later ImportError easy to debug
import importlib.metadata as _md
for _p in ["transformers", "accelerate", "bitsandbytes", "torch",
           "langchain-core", "langchain-community", "langchain-huggingface",
           "langchain-text-splitters", "faiss-cpu", "sentence-transformers"]:
    try:
        print(f"{_p:24s} {_md.version(_p)}")
    except Exception:
        print(f"{_p:24s} NOT INSTALLED")

print("\nIf Kaggle shows a 'restart' prompt after this cell, restart the kernel and "
      "run again from here.")

## 2. Check the GPU & make long output wrap

Kaggle free GPUs: **T4** (16 GB, fast fp16) or **P100** (16 GB, slower fp16). Either is
fine for a 4-bit 4B model. We also switch on line-wrapping so long answers stay on
screen when projected.

In [ ]:
# (1) CSS: wrap streamed text output instead of letting it scroll off the page
from IPython.display import HTML, display
display(HTML(
    "<style>"
    "pre, .output pre, .jp-RenderedText pre, .output-plaintext "
    "{ white-space: pre-wrap !important; word-break: break-word; }"
    "</style>"
))

# (2) helper: print a long string wrapped to a fixed column width
import textwrap
def show(text, width=100):
    for line in (str(text).splitlines() or [""]):
        print(textwrap.fill(line, width=width, replace_whitespace=False) if line else "")

# (3) quiet transformers' non-fatal chatter (e.g. the harmless
#     "Both max_new_tokens and max_length seem to have been set" notice that prints
#     on every generation). Comment this out if you want to see all warnings.
import transformers
transformers.logging.set_verbosity_error()
import warnings
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")

import torch
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  |  {p.total_memory/1024**3:.1f} GB  |  cc {p.major}.{p.minor}")
    print("bf16 supported :", torch.cuda.is_bf16_supported())   # False on T4/P100
    !nvidia-smi
else:
    print(
        "\n"
        "############################################################\n"
        "  NO GPU ATTACHED - the model cells below will NOT run.\n"
        "  1. right panel -> Settings -> Accelerator -> GPU T4 x2\n"
        "  2. wait for confirmation, then Run -> Restart & clear outputs\n"
        "  3. re-run from section 1\n"
        "  (If it still says no GPU, your weekly Kaggle GPU quota is used\n"
        "   up - check Settings; it resets weekly.)\n"
        "############################################################"
    )

## 3. The LangChain pieces we use

| Object | Role |
|--------|------|
| `Document` | text + `metadata` dict |
| `WebBaseLoader` | URL → `Document` |
| `RecursiveCharacterTextSplitter` | long `Document` → smaller chunks |
| `HuggingFaceEmbeddings` | text → vector |
| `FAISS` | store + similarity-search the vectors |
| `retriever` = `vectorstore.as_retriever()` | question → top-k `Document`s |
| `ChatPromptTemplate` | the prompt with `{context}` / `{input}` slots |
| `HuggingFacePipeline` | wraps the 🤗 text-generation pipeline as a Runnable |
| LCEL (`|`, `RunnablePassthrough.assign`) | glue everything into one chain |

Because `Qwen3` is a *reasoning* model, we add one small custom step (section 4) to
turn it OFF for this demo — the standard chat wrapper doesn't expose that switch.

## 4. Load `Qwen/Qwen3-4B` in 4-bit and build a chat Runnable

Qwen3-4B in fp16 needs ~8 GB just for weights, plus activations and the
KV-cache during generation. 4-bit **NF4** quantization (`bitsandbytes`) stores the
weights in ~3 GB with a small quality cost, leaving comfortable headroom on a 16 GB
T4 / P100. Compute still happens in fp16.

**Qwen3 "thinking" mode.** Qwen3 can emit a hidden `<think> … </think>` reasoning
block before its answer. For a RAG demo we want just the grounded answer, so we call
the chat template with **`enable_thinking=False`** and also strip any stray think
tags. `langchain-huggingface`'s `ChatHuggingFace` can't pass that flag, so we do the
chat-templating ourselves in a tiny `RunnableLambda` — everything else stays LCEL.

> **This cell needs a GPU.** 4-bit `bitsandbytes` only runs on CUDA. If you see
> `RuntimeError: Cannot access accelerator device when none is available` it means no
> GPU is attached — see the guard message below and fix the Settings panel.

In [ ]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, set_seed
from langchain_huggingface import HuggingFacePipeline
from langchain_core.runnables import RunnableLambda

# ---- hard stop with a clear message if there is no GPU ----------------------------
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available, so 4-bit Qwen3-4B cannot load.\n"
        "Fix it in the right-hand panel:\n"
        "  1. Settings -> Accelerator -> GPU T4 x2  (or GPU P100)\n"
        "  2. wait for 'GPU is now available', then  Run -> Restart & clear cell outputs\n"
        "  3. re-run from section 1\n"
        "Also check Settings -> your GPU quota for the week isn't used up "
        "(Kaggle silently falls back to CPU when it is)."
    )
print("GPU OK:", torch.cuda.get_device_name(0))

MODEL_NAME = "Qwen/Qwen3-4B"

# 4-bit quantization config: NF4 + double quantization, fp16 math (T4/P100 have no bf16)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},          # whole model on GPU 0 (don't split across a T4 pair)
    torch_dtype=torch.float16,
)
base_model.eval()

# The default generation_config ships with max_length=20. We always pass
# max_new_tokens instead, so clear max_length to silence the
# "Both max_new_tokens and max_length seem to have been set" warning on every call.
base_model.generation_config.max_length = None

print("Loaded", MODEL_NAME, "in 4-bit on", next(base_model.parameters()).device)


# --- turn LangChain messages (or a plain string) into a Qwen3 prompt string ---
_LC_ROLE = {"system": "system", "human": "user", "ai": "assistant"}

def to_qwen_prompt(value, enable_thinking=False):
    # `value` is a plain str (single user turn) OR a ChatPromptValue / list of messages
    if isinstance(value, str):
        conversation = [{"role": "user", "content": value}]
    else:
        messages = value.to_messages() if hasattr(value, "to_messages") else value
        conversation = [{"role": _LC_ROLE.get(m.type, "user"), "content": m.content} for m in messages]
    # enable_thinking=False  ->  Qwen3 answers directly, no <think> block
    return tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    )

_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)

def strip_think(text):
    # safety net in case a think block slips through
    return _THINK_RE.sub("", text).replace("<think>", "").replace("</think>", "").strip()


def build_qwen_chat(max_new_tokens=400, do_sample=True, temperature=0.7,
                    top_p=0.8, top_k=20, repetition_penalty=1.1):
    # Build a LangChain Runnable:  (messages|str) -> Qwen prompt -> generate -> clean text
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        repetition_penalty=repetition_penalty,
        return_full_text=False,                       # give back only the new text
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    if do_sample:                                     # these only apply when sampling
        gen_kwargs.update(temperature=temperature, top_p=top_p, top_k=top_k)

    text_pipeline = pipeline("text-generation", model=base_model, tokenizer=tokenizer, **gen_kwargs)
    # make sure the pipeline's own generation_config doesn't also carry max_length=20
    text_pipeline.model.generation_config.max_length = None
    hf_runnable = HuggingFacePipeline(pipeline=text_pipeline)

    return RunnableLambda(to_qwen_prompt) | hf_runnable | RunnableLambda(strip_think)


# a general chat model for the "no RAG" and creativity demos
chat_model = build_qwen_chat(temperature=0.7, top_p=0.8, top_k=20)

set_seed(0)
print("\nSanity check:", chat_model.invoke("Reply with exactly: RAG demo ready."))

## 5. `temperature`, `top_p`, `top_k` — what they do

At each step the model gives a probability to **every** vocabulary token. These knobs
decide how the next token is picked:

| Setting | Meaning | Effect |
|---------|---------|--------|
| `do_sample=False` (greedy) | always take the single most likely token | fully deterministic, can be flat |
| `temperature` | divides the logits before softmax: `<1` sharpens (safer), `>1` flattens (wilder), `→0` ≈ greedy | higher = more random |
| `top_k` | keep only the `k` most-likely tokens, then sample | small `k` = safer vocabulary |
| `top_p` (nucleus) | keep the smallest group of tokens whose probabilities sum to `p` | adapts to the model's confidence |
| `repetition_penalty` | down-weights tokens already used | `>1` avoids "the the the" loops |
| `max_new_tokens` | generation length cap | speed / cost control |

**For RAG we use a low temperature** so the answer sticks to the retrieved facts and
is repeatable. **For creative writing** you'd raise it. Watch the spread below — the
prompt never changes, only the knobs.

In [ ]:
CREATIVE_PROMPT = "In one vivid sentence, describe a river flowing through a mountain valley."

def compare_runs(chat, label, seeds=(1, 2)):
    outs = []
    for s in seeds:
        set_seed(s)
        outs.append(chat.invoke(CREATIVE_PROMPT))
    print("=" * 95)
    print(label)
    print("-" * 95)
    for k, o in enumerate(outs, 1):
        show(f"run {k}: {o}")

tiny = dict(max_new_tokens=45)

compare_runs(build_qwen_chat(do_sample=False, **tiny),
             "GREEDY (do_sample=False)  -> identical every run")
compare_runs(build_qwen_chat(temperature=0.2, top_k=0, top_p=1.0, **tiny),
             "temperature = 0.2         -> small, safe variation")
compare_runs(build_qwen_chat(temperature=1.3, top_k=0, top_p=1.0, **tiny),
             "temperature = 1.3         -> wild, very different each run")
compare_runs(build_qwen_chat(temperature=1.0, top_k=5,  top_p=1.0, **tiny),
             "top_k = 5                 -> only the 5 likeliest tokens are eligible")
compare_runs(build_qwen_chat(temperature=1.0, top_k=0,  top_p=0.5, **tiny),
             "top_p = 0.5 (nucleus)     -> tight nucleus, focused wording")

## 6. Ask Qwen3-4B **without** RAG

No article supplied — the model answers from pretraining only.

In [ ]:
QUESTION = "What happened in the recent glacier collapse in Nepal in August 2026?"

set_seed(0)
no_rag_reply = chat_model.invoke(QUESTION)

print("QUESTION:", QUESTION)
print("\n--- answer from pretrained knowledge only ---\n")
show(no_rag_reply)

The model's knowledge stops at its training cut-off, so a very recent event may come
back vague, incomplete, or partly invented. We just observe the response — we don't
claim it knows nothing.

## 7. Load the news page with `WebBaseLoader`

`WebBaseLoader` downloads the URL and, via a BeautifulSoup `SoupStrainer`, keeps only
`<p>` / heading / list text — dropping scripts, styles, nav bars and menus.

`NEWS_URL` is a real Al Jazeera article. If Kaggle can't reach it, the code falls back
to the Wikipedia page on the same event. **If both fail, check Internet is ON.**

In [ ]:
import os
os.environ["USER_AGENT"] = "Mozilla/5.0 (classroom-rag-demo; educational use)"

import bs4
from langchain_community.document_loaders import WebBaseLoader

NEWS_URL     = "https://www.aljazeera.com/news/2026/8/27/nepal-tibet-floods-what-happened-what-caused-them-and-who-is-missing"
FALLBACK_URL = "https://en.wikipedia.org/wiki/2026_Nepal_floods"

article_strainer = bs4.SoupStrainer(["p", "h1", "h2", "h3", "li"])

def load_web_article(url):
    loader = WebBaseLoader(
        web_paths=[url],
        bs_kwargs={"parse_only": article_strainer},
        requests_kwargs={"timeout": 30},
    )
    docs = loader.load()
    for d in docs:                       # tidy up blank lines the strainer leaves behind
        d.page_content = "\n".join(ln.strip() for ln in d.page_content.splitlines() if ln.strip())
    return docs

try:
    web_docs = load_web_article(NEWS_URL)
    if len(web_docs[0].page_content.split()) < 200:
        raise ValueError("too little text returned (site may be blocking us)")
    SOURCE_URL = NEWS_URL
except Exception as e:
    print("Primary URL unusable:", e, "\n-> falling back to Wikipedia\n")
    web_docs = load_web_article(FALLBACK_URL)
    SOURCE_URL = FALLBACK_URL

print("Source used :", SOURCE_URL)
print("Word count  :", len(web_docs[0].page_content.split()))
print("Metadata    :", web_docs[0].metadata)
print("\n---------------- PREVIEW (first 1200 chars) ----------------\n")
show(web_docs[0].page_content[:1200])

## 8. Split into chunks — `RecursiveCharacterTextSplitter`

Retrieval works best on small, focused passages. The splitter tries paragraph breaks
first, then sentences, then words. `chunk_overlap` repeats a little text across each
boundary so a fact split between two chunks still appears whole in one of them.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,        # characters, not words (~2-3 paragraphs)
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(web_docs)

print("Number of chunks:", len(chunks))
print("\n---------------- SAMPLE CHUNK [1] ----------------\n")
show(chunks[1].page_content)

## 9. Embeddings — `HuggingFaceEmbeddings`

An embedding maps text to a vector so that similar meanings are close together — that
is what lets us retrieve by meaning instead of keywords. Model:
`sentence-transformers/all-MiniLM-L6-v2` (tiny, fast, 384-dim). We normalise the
vectors so "closeness" ≈ cosine similarity.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

demo_vec = embeddings.embed_query("glacier collapse in Nepal")
print("Embedding dim :", len(demo_vec))
print("First 8 values:", [round(x, 4) for x in demo_vec[:8]])

## 10. FAISS vector store

FAISS stores the chunk vectors in memory and searches them fast — no server, no cloud.
`FAISS.from_documents` embeds every chunk and indexes it in one call.

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)
print("Vectors in the FAISS index:", vectorstore.index.ntotal)

## 11. Retriever + a look at the scores

`as_retriever()` gives a `question → top-k Documents` component. We also call
`similarity_search_with_score` directly so we can see the FAISS **L2 distances**
(smaller = more similar).

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

probe = "What caused the disaster?"
print("QUESTION:", probe)
print("\n--- retrieved chunks + FAISS L2 distance (lower = closer) ---\n")
for rank, (doc, score) in enumerate(vectorstore.similarity_search_with_score(probe, k=3), start=1):
    print(f"[#{rank}] distance = {score:.3f}")
    show(doc.page_content[:300].strip() + " ...")
    print()

## 12. The RAG prompt — `ChatPromptTemplate`

`{context}` will be filled with the retrieved chunks, `{input}` with the question.
The system message pins the model to the supplied context.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_RULES = (
    "You are a factual assistant for a classroom demo. "
    "Answer the question using ONLY the context provided below. "
    "If the answer is not in the context, reply exactly: "
    "'The information is not available in the provided article.' "
    "Be concise and do not add outside knowledge."
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_RULES),
    ("human", "Context from the news article:\n\n{context}\n\nQuestion: {input}"),
])

print(rag_prompt.format(context="<retrieved chunks>", input="<question>"))

## 13. Build the RAG chain (pure LCEL)

Each `RunnablePassthrough.assign(...)` **adds a key** to the dict flowing through, so
every intermediate value survives and we can print it:

```
{"input": q}
  .assign(docs    = retrieve top-k chunks)
  .assign(context = format the chunks into one text block)
  .assign(answer  = rag_prompt | qwen(4-bit, low temp) )
```

The answer model uses **temperature 0.1** for grounded, repeatable answers.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

answer_model = build_qwen_chat(temperature=0.1, top_p=0.9, top_k=20, max_new_tokens=350)

def format_docs(docs):
    return "\n\n".join(f"[chunk {i}] {d.page_content}" for i, d in enumerate(docs, start=1))

# rag_prompt gets the whole dict and just picks {context} and {input} out of it
generate_answer = rag_prompt | answer_model     # answer_model already returns clean text

rag_chain = (
    RunnablePassthrough.assign(docs=lambda x: retriever.invoke(x["input"]))
    | RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))
    | RunnablePassthrough.assign(answer=generate_answer)
)

def ask_rag(question, seed=0, show_chunks=True):
    set_seed(seed)
    result = rag_chain.invoke({"input": question})
    print("QUESTION:", question)
    if show_chunks:
        print("\n-- retrieved chunks (what the LLM was given) --")
        for i, d in enumerate(result["docs"], start=1):
            show(f"[chunk {i}] {d.page_content[:220].strip()} ...")
    print("\n-- grounded answer --")
    show(result["answer"])
    print()
    return result

_ = ask_rag("What caused the disaster?")

## 14. Run RAG on several questions

**Question → retrieved chunks → grounded answer**, each time.

In [ ]:
for q in [
    "What caused the disaster?",
    "Which areas were affected?",
    "What happened after the glacier collapse?",
    "What risks remain?",
    "What are the authorities doing?",
]:
    print("#" * 95)
    ask_rag(q)

## 15. Side by side — without RAG vs with RAG

In [ ]:
compare_q = "What happened in the recent glacier collapse in Nepal in August 2026?"

print("=" * 95); print("A)  Qwen3-4B ALONE  (no retrieval)"); print("=" * 95)
set_seed(0)
show(chat_model.invoke(compare_q))

print("\n" + "=" * 95); print("B)  Qwen3-4B + RAG  (same question + retrieved chunks)"); print("=" * 95)
set_seed(0)
show(rag_chain.invoke({"input": compare_q})["answer"])

RAG did **not** change the model's weights. The retrieved chunks were placed in the
prompt at inference time; remove retrieval and you get answer A back.

## 16. Measure the performance (this is what you came to test)

How fast is 4-bit Qwen3-4B on this Kaggle GPU, and how much memory does it use?

**Rough expectations for 4-bit `bitsandbytes`:** T4 ≈ 12–25 tokens/sec, P100 a bit
slower. If that feels too slow for a live class, options are: use `Qwen2.5-3B-Instruct`
in fp16, lower `max_new_tokens`, or reduce `k`.

In [ ]:
import time

perf_q = "Summarize what happened, its cause, and the main risks, in about 5 sentences."
docs = retriever.invoke(perf_q)
prompt_value = rag_prompt.invoke({"context": format_docs(docs), "input": perf_q})

prompt_text = to_qwen_prompt(prompt_value)
n_prompt_tokens = len(tokenizer(prompt_text).input_ids)

torch.cuda.reset_peak_memory_stats()
set_seed(0)
t0 = time.time()
answer = answer_model.invoke(prompt_value)
elapsed = time.time() - t0

n_gen_tokens = len(tokenizer(answer).input_ids)

print(f"prompt length      : {n_prompt_tokens} tokens")
print(f"generated          : ~{n_gen_tokens} tokens")
print(f"generation time    : {elapsed:.1f} s")
print(f"throughput         : ~{n_gen_tokens / max(elapsed, 1e-6):.1f} tokens/sec")
print(f"peak GPU memory    : {torch.cuda.max_memory_allocated() / 1e9:.2f} GB "
      f"(weights + activations + KV-cache)")
print("\n--- the answer it produced ---\n")
show(answer)

## 17. The minimal chain — when you only want the answer string

Section 13 kept the chunks for display. If you only need the answer, the whole thing
is three pipes:

In [ ]:
minimal_rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | rag_prompt
    | answer_model
)

set_seed(0)
show(minimal_rag_chain.invoke("Which areas were affected?"))

## 18. Summary

**Traditional LLM**

```
Question ──► LLM ──► Answer
```

**RAG**

```
Question ─► Retriever ─► relevant chunks from the web page ─► Prompt ─► LLM ─► Grounded answer
```

**RAG does not update the model's weights.** External text is retrieved and supplied as
context at inference time.
